# Que prédit notre classifieur ?

Lire les scores, repérer l’erreur et prévoir l’effet d’une seule retouche.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#du-pipeline-au-modele).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Nous allons lire une réponse avant de chercher à la calculer. Pour chaque extrait, imaginons deux scores : le premier pour voix, le second pour percussion. Plus le score est grand, plus cette réponse est préférée.

| Extrait | Score voix, colonne 0 | Score percussion, colonne 1 | Annotation |
| --- | --- | --- | --- |
| A | 1,5 | −1,5 | Voix, 0 |
| B | 4,5 | −4,5 | Percussion, 1 |
| C | 2,5 | −2,5 | Voix, 0 |

Ces nombres sont choisis à la main pour apprendre à lire une sortie. Au chapitre suivant, une couche les calculera depuis X.

Un score brut, aussi appelé logit, peut être négatif ou dépasser un. Il indique ici une préférence relative, pas encore une probabilité. Pour décider, il suffit de trouver la colonne qui gagne dans chaque ligne.

argmax renvoie la position du maximum. dim=1 compare les colonnes, donc les deux classes de chaque extrait. Nous obtenons [0,0,0] : le modèle répond voix trois fois. Le test == compare ensuite chaque décision à son annotation : [True,False,True].

C’est bien B qui pose problème. Il ne faut pas réordonner les annotations pour obtenir un accord : ce serait changer la question. Chaque réponse reste attachée à l’extrait dont elle provient.

Pouvons-nous corriger uniquement B en retouchant ce tableau ? Essayons de remplacer son score percussion par 5, sans toucher aux deux autres lignes. Avant le résultat, repérez précisément la case concernée.

Nous gardons aussi le premier tableau pour comparer. clone() copie ses valeurs dans un stockage distinct. L’affectation [1,1] = 5 écrit ensuite dans cette copie : ligne de B, colonne percussion.

Le changement fonctionne parce que 5 dépasse 4,5. A et C conservent leur décision ; le tableau initial garde aussi son ancien score −4,5. Cette retouche illustre le choix d’une classe, mais ne fournit aucune règle pour un nouvel extrait.

Nous ne voulons pas corriger une case à la main à chaque écoute. La prochaine étape sera de calculer les scores à partir des mesures, avec des poids que nous pourrons ensuite ajuster.

## Démonstration 1 · Exécuter l’expérience entière

Retrouver les préparations et suivre les valeurs dans l’ordre du programme.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

Une ligne contient deux scores et correspond à une annotation. Les scores sont ici imposés pour la démonstration.

In [4]:
scores = torch.tensor([[1.5, -1.5], [4.5, -4.5], [2.5, -2.5]])
cibles = torch.tensor([0, 1, 0], dtype=torch.long)


In [5]:
observer('scores', scores)
observer('cibles', cibles)

scores forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[1.5, -1.5], [4.5, -4.5], [2.5, -2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]


### Calculer l’état suivant

argmax réduit les colonnes en une position gagnante par ligne. Le test == garde un booléen par extrait.

In [6]:
predictions = scores.argmax(dim=1)
accords = predictions == cibles


In [7]:
observer('predictions', predictions)
observer('accords', accords)

predictions forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 0, 0]
accords forme (3,) type torch.bool requires_grad False
  valeurs : [True, False, True]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [8]:
print("decisions :", predictions.tolist())
print("accords :", accords.tolist())


decisions : [0, 0, 0]
accords : [True, False, True]


### Calculer l’état suivant

clone crée un stockage indépendant. Seule la case ligne 1, colonne 1 de la copie reçoit 5.

In [9]:
scores_modifies = scores.clone()
scores_modifies[1, 1] = 5.


In [10]:
observer('scores_modifies', scores_modifies)

scores_modifies forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[1.5, -1.5], [4.5, 5.0], [2.5, -2.5]]


### Calculer l’état suivant

La même réduction compare maintenant les valeurs modifiées. Le tableau original permet de vérifier que la copie était distincte.

In [11]:
predictions_modifiees = scores_modifies.argmax(dim=1)


In [12]:
observer('predictions_modifiees', predictions_modifiees)

predictions_modifiees forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [13]:
print("decisions apres modification :", predictions_modifiees.tolist())
print("ancien score percussion B :", scores[1, 1].item())

decisions apres modification : [0, 1, 0]
ancien score percussion B : -4.5


In [14]:
assert bool(predictions.tolist() == [0, 0, 0] and accords.tolist() == [True, False, True]), 'du-pipeline-au-modele/main: predictions.tolist() == [0, 0, 0] and accords.tolist() == [True, False, True]'
assert bool(predictions_modifiees.tolist() == [0, 1, 0]), 'du-pipeline-au-modele/main: predictions_modifiees.tolist() == [0, 1, 0]'
assert bool(scores[1, 1].item() == -4.5 and scores_modifies[1, 1].item() == 5.), 'du-pipeline-au-modele/main: scores[1, 1].item() == -4.5 and scores_modifies[1, 1].item() == 5.'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Prévoyez les indices de classes renvoyés par argmax sur les colonnes.

In [15]:
exo_scores = torch.tensor([[1.5,-1.5],[4.5,-4.5],[2.5,-2.5]])
exo_predictions = ...  # Remplacer les points par votre réponse.
if exo_predictions is Ellipsis or exo_predictions is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    torch.testing.assert_close(torch.as_tensor(exo_predictions),exo_scores.argmax(dim=1))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Les trois lignes ont leur maximum en colonne 0. La taille du lot n’est pas le nombre de classes.

</details>

### Exercice 2

Reliez chaque décision à sa cible : quelles lignes sont correctement classées ?

In [16]:
exo_predictions = torch.tensor([0,0,0])
exo_cibles = torch.tensor([0,1,0])
exo_accords = ...  # Remplacer les points par votre réponse.
if exo_accords is Ellipsis or exo_accords is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.equal(torch.as_tensor(exo_accords),exo_predictions == exo_cibles)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Comparez les deux vecteurs position par position, en conservant l’ordre des extraits.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_scores = torch.tensor([[1.5,-1.5],[4.5,-4.5],[2.5,-2.5]])
exo_predictions = torch.tensor([0,0,0])
if exo_predictions is Ellipsis or exo_predictions is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    torch.testing.assert_close(torch.as_tensor(exo_predictions),exo_scores.argmax(dim=1))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_predictions = torch.tensor([0,0,0])
exo_cibles = torch.tensor([0,1,0])
exo_accords = torch.tensor([True,False,True])
if exo_accords is Ellipsis or exo_accords is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.equal(torch.as_tensor(exo_accords),exo_predictions == exo_cibles)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [19]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()